# Analisis exploratorio - Art Institute of Chicago

Este notebook lee los datos crudos guardados en MongoDB desde la coleccion `raw_data` de la base de datos `taller4_db`. Luego selecciona variables relevantes para construir un DataFrame y generar insights y visualizaciones.

## 1. Importar librerias y conectar a MongoDB

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
MONGO_URI = "mongodb://localhost:27017"
MONGO_DB = "taller4_db"
MONGO_COLLECTION = "raw_data"

client = MongoClient(MONGO_URI)
collection = client[MONGO_DB][MONGO_COLLECTION]

records = list(collection.find({}, {"_id": 0}))
print(f"Documentos cargados desde MongoDB: {len(records)}")

if len(records) < 100:
    raise ValueError("Se requieren al menos 100 registros. Ejecuta primero python ingesta.py")

## 2. Crear DataFrame con variables seleccionadas

Se seleccionan variables relacionadas con identificacion de la obra, artista, fecha, departamento, tipo de obra, origen y medio.

In [ ]:
raw_df = pd.DataFrame(records)

columns = [
    "id",
    "title",
    "artist_title",
    "date_start",
    "department_title",
    "artwork_type_title",
    "place_of_origin",
    "medium_display",
]

df = raw_df[columns].copy()
df.head()

## 3. Limpieza basica

In [ ]:
df["artist_title"] = df["artist_title"].fillna("Artista no identificado")
df["department_title"] = df["department_title"].fillna("Sin departamento")
df["artwork_type_title"] = df["artwork_type_title"].fillna("Sin tipo")
df["place_of_origin"] = df["place_of_origin"].fillna("Origen no identificado")
df["medium_display"] = df["medium_display"].fillna("Medio no identificado")
df["date_start"] = pd.to_numeric(df["date_start"], errors="coerce")

df.head()

## 4. Inspeccion basica

Se revisan las primeras filas, los tipos de datos y la cantidad de valores nulos por columna.

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 5. Insights numericos y conteos

In [ ]:
total_obras = len(df)
artistas_unicos = df["artist_title"].nunique()
departamento_top = df["department_title"].value_counts().idxmax()
cantidad_departamento_top = df["department_title"].value_counts().max()
tipo_top = df["artwork_type_title"].value_counts().idxmax()
cantidad_tipo_top = df["artwork_type_title"].value_counts().max()
origen_top = df["place_of_origin"].value_counts().idxmax()
cantidad_origen_top = df["place_of_origin"].value_counts().max()

years = df["date_start"].dropna()
anio_min = int(years.min()) if not years.empty else "No disponible"
anio_max = int(years.max()) if not years.empty else "No disponible"

insights = [
    f"El conjunto analizado contiene {total_obras} obras de arte.",
    f"Hay {artistas_unicos} artistas o autores distintos en los registros seleccionados.",
    f"El departamento con mas obras es '{departamento_top}', con {cantidad_departamento_top} registros.",
    f"El tipo de obra mas frecuente es '{tipo_top}', con {cantidad_tipo_top} registros.",
    f"El lugar de origen mas frecuente es '{origen_top}', con {cantidad_origen_top} registros.",
    f"Las fechas de inicio de las obras van desde {anio_min} hasta {anio_max}.",
]

for number, insight in enumerate(insights, start=1):
    print(f"{number}. {insight}")

## 6. Grafico de torta obligatorio

El grafico muestra la proporcion de obras por departamento. Para mantenerlo legible, se agrupan los departamentos menos frecuentes en la categoria `Otros`.

In [ ]:
department_counts = df["department_title"].value_counts()
top_departments = department_counts.head(5)
other_departments = department_counts.iloc[5:].sum()

pie_data = top_departments.copy()
if other_departments > 0:
    pie_data.loc["Otros"] = other_departments

plt.figure(figsize=(8, 8))
plt.pie(pie_data, labels=pie_data.index, autopct="%1.1f%%", startangle=90)
plt.title("Proporcion de obras por departamento")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 7. Grafico libre 1: barras por tipo de obra

In [ ]:
type_counts = df["artwork_type_title"].value_counts().head(10)

sns.barplot(x=type_counts.values, y=type_counts.index, hue=type_counts.index, palette="viridis", legend=False)
plt.title("Top 10 tipos de obra")
plt.xlabel("Cantidad de obras")
plt.ylabel("Tipo de obra")
plt.tight_layout()
plt.show()

## 8. Grafico libre 2: distribucion de fechas

In [ ]:
valid_years = df["date_start"].dropna()

sns.histplot(valid_years, bins=20, kde=True, color="#2a9d8f")
plt.title("Distribucion de fechas de inicio de las obras")
plt.xlabel("Anio de inicio")
plt.ylabel("Cantidad de obras")
plt.tight_layout()
plt.show()

## 9. Conclusion preliminar

Con este analisis se puede observar la composicion inicial de las obras segun departamento, tipo de obra, origen y fechas. Estos resultados sirven como base para redactar los 5 insights finales en el PDF de entrega.